# 3D Engine — TRELLIS.2 → Unreal-safe GLB

**Independent engine:** reference image → PBR 3D asset.

This engine handles objects, props, environments and characters. Nothing enters the Animation Engine automatically.

**Important contract:** TRELLIS geometry is normalized. Before export you must give the asset one real-world dimension (for example: character height 1.75 m, chair height 0.9 m, door height 2.0 m). The notebook bakes that scale into the GLB and writes a JSON manifest.


In [ ]:
import shutil, subprocess

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True, capture_output=True, check=True
).stdout.strip().splitlines()
if not smi:
    raise RuntimeError("No NVIDIA GPU detected. In Colab choose Runtime → Change runtime type → GPU.")

gpu_name, memory_mib = [x.strip() for x in smi[0].rsplit(",", 1)]
memory_mib = int(memory_mib)
free_gib = shutil.disk_usage("/content").free / (1024**3)
print(f"GPU: {gpu_name} | VRAM: {memory_mib/1024:.1f} GiB | Free disk: {free_gib:.1f} GiB")
if memory_mib < 24000:
    raise RuntimeError("Use a >=24 GB NVIDIA GPU for the supported TRELLIS.2 path.")
if free_gib < 35:
    raise RuntimeError("Need at least 35 GiB free disk.")


In [ ]:
import pathlib, shutil
if pathlib.Path("/content/My-works").exists():
    shutil.rmtree("/content/My-works")
!git clone -q --depth 1 https://github.com/Logan17de/My-works.git /content/My-works
TOOLS = "/content/My-works/ai-3d-animation-engines/3d-engine"
print("Helpers:", TOOLS)


## Install pinned TRELLIS.2

The runtime is disposable. This recreates the environment cleanly each time.


In [ ]:
%%bash
set -euo pipefail

TRELLIS_REF="75fbf0183001ed9876c8dbb35de6b68552ee08bd"

apt-get update -qq
apt-get install -y -qq git git-lfs build-essential cmake ninja-build wget ffmpeg sudo \
    libjpeg-dev libgl1-mesa-dev libegl1-mesa-dev

if [ ! -x /opt/conda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p /opt/conda
fi
source /opt/conda/etc/profile.d/conda.sh

rm -rf /content/TRELLIS.2 /tmp/extensions
git clone -q --recursive https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS.2
git -C /content/TRELLIS.2 checkout -q "$TRELLIS_REF"
git -C /content/TRELLIS.2 submodule sync --recursive
git -C /content/TRELLIS.2 submodule update --init --recursive --force

conda env remove -n trellis2 -y >/dev/null 2>&1 || true
conda create -n trellis2 python=3.10 -y -q
conda activate trellis2
python -m pip install -q --upgrade pip setuptools wheel packaging ninja
python -m pip install torch==2.6.0 torchvision==0.21.0 \
    --index-url https://download.pytorch.org/whl/cu124

if [ -x /usr/local/cuda-12.4/bin/nvcc ]; then
  export CUDA_HOME=/usr/local/cuda-12.4
else
  conda install -y -q -c nvidia/label/cuda-12.4.1 cuda-toolkit
  export CUDA_HOME="$CONDA_PREFIX"
fi
export PATH="$CUDA_HOME/bin:$PATH"

cd /content/TRELLIS.2
. ./setup.sh --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm

python - <<'PY'
import torch, o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline
assert torch.cuda.is_available()
print("TRELLIS.2 smoke test: OK", torch.__version__, torch.version.cuda)
PY


In [ ]:
from google.colab import files
import pathlib

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one reference image.")
input_name = next(iter(uploaded))
INPUT_IMAGE = f"/content/{input_name}"
if pathlib.Path(INPUT_IMAGE).suffix.lower() not in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
    raise ValueError("Expected PNG/JPG/JPEG/WEBP/BMP.")
print("Input:", INPUT_IMAGE)


## Set the asset's real-world scale

Edit these values before generation.

Examples:
- character: `height`, `1.75`
- door: `height`, `2.0`
- sofa: `width`, `2.0`
- small prop with uncertain orientation: `longest`, approximate size


In [ ]:
ASSET_TYPE = "object" #@param ["object", "character", "environment", "other"]
TARGET_AXIS = "longest" #@param ["width", "height", "depth", "longest"]
TARGET_SIZE_METERS = 1.0 #@param {type:"number"}

if TARGET_SIZE_METERS <= 0:
    raise ValueError("TARGET_SIZE_METERS must be positive.")
print(f"Scale contract: {TARGET_AXIS} = {TARGET_SIZE_METERS} m | type={ASSET_TYPE}")


In [ ]:
import subprocess, pathlib, shlex, json

OUTPUT_DIR = "/content/trellis_outputs"
ASSET_NAME = "asset"

cmd = [
    "/opt/conda/bin/conda", "run", "-n", "trellis2",
    "python", f"{TOOLS}/run_trellis2.py",
    "--input", INPUT_IMAGE,
    "--output-dir", OUTPUT_DIR,
    "--name", ASSET_NAME,
    "--asset-type", ASSET_TYPE,
    "--target-axis", TARGET_AXIS,
    "--target-size-m", str(TARGET_SIZE_METERS),
    "--envmap", "/content/TRELLIS.2/assets/hdri/forest.exr",
    "--decimation-target", "1000000",
    "--texture-size", "4096",
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd="/content/TRELLIS.2", check=True)

GLB_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}.glb"
MANIFEST_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}_manifest.json"
PREVIEW_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}_preview.mp4"

for required in (GLB_PATH, MANIFEST_PATH):
    if not pathlib.Path(required).is_file():
        raise RuntimeError(f"Expected output missing: {required}")

contract = json.loads(pathlib.Path(MANIFEST_PATH).read_text())
print(json.dumps(contract["geometry"], indent=2))


In [ ]:
from IPython.display import Video, display
import pathlib

if pathlib.Path(PREVIEW_PATH).is_file():
    display(Video(PREVIEW_PATH, embed=True))
else:
    print("No preview MP4; GLB and manifest are still valid.")


In [ ]:
from google.colab import files
files.download(GLB_PATH)
files.download(MANIFEST_PATH)


### Output contract

Keep **both** files:
- `asset.glb` — PBR geometry at explicit real-world scale, with standard GLB textures (no required WebP extension).
- `asset_manifest.json` — dimensions, scale and downstream compatibility metadata.

For a humanoid, manually choose `asset.glb` later in the Animation Engine.
